# Out-of-sample phenotype estimation

This notebook evaluates split-half PC-score agreement in participants excluded from phenotype-model fitting. Scaling parameters, the MMER-XGBoost model, covariance components, and PC axes are learned within each training fold and frozen for held-out scoring.

## Notebook flow

1. Create five participant-level cross-fitting folds.
2. Fit and freeze the phenotype model using free-living meals from each training fold.
3. Estimate paired PC scores from odd/even and chronological meal halves in held-out participants.
4. Inspect held-out score tables and fold-level fit diagnostics.
5. Export pooled and fold-specific ICC and correlation summaries for both split designs.

## Outputs

CSV files written to `Results/OOS_PC_estimation/`:

| File | Contents |
| --- | --- |
| `oos_icc_odd_even_split_half_summary.csv` | Odd/even PC1/PC2 ICC and correlation summaries, by fold and pooled. |
| `oos_icc_odd_even_split_half_full.csv` | Complete odd/even ICC statistics, including all computed ICC variants. |
| `oos_icc_temporal_split_half_summary.csv` | Chronological PC1/PC2 ICC and correlation summaries, by fold and pooled. |
| `oos_icc_temporal_split_half_full.csv` | Complete chronological ICC statistics, including all computed ICC variants. |

## 1. Setup

Resolve project paths, import the cross-fitting utilities, and load the common prepared meal-level dataset.

In [1]:
from pathlib import Path
import sys

# Find the project when launched from its root, Code/, or a notebook subfolder.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CODE_DIR = PROJECT_ROOT / "Code"
    if (CODE_DIR / "data_paths.py").is_file():
        break
else:
    raise FileNotFoundError("Open this notebook from within the project directory.")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared defaults; override individual paths here if needed.
from data_paths import DATA_DIR, METADATA_PATH, MEAL_DATA_PATH, CGM_METRICS_PATH
FIGURES_DIR = PROJECT_ROOT / "Figures"
RESULTS_DIR = PROJECT_ROOT / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "OOS_PC_estimation").mkdir(parents=True, exist_ok=True)


In [2]:
import logging
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import xgboost as xgb
from mmer import MixedEffectEstimator
from sklearn.model_selection import BaseCrossValidator, GroupKFold
from tqdm import tqdm

In [3]:
from utils import (
    DATE_COL,
    OUTCOMES,
    RANDOM_SLOPES,
    SUBJECT_COL,
    build_design_mats,
    derive_pc_basis,
    fit_training_fold,
    icc_split_half,
    infer_blups_from_frozen_model,
    load_and_prepare_data,
    make_subject_folds,
    make_xgb,
    project_blups_to_pcs,
    split_half_within_subject,
    transform_validation_data,
)

In [4]:
meta_data, data, id_to_subject_key, _= load_and_prepare_data(
    METADATA_PATH,
    MEAL_DATA_PATH,
)

print(f"Prepared {len(data):,} meals from {data['subject_key'].nunique():,} participants.")


data shape : (54987, 133)
Prepared 54,987 meals from 992 participants.


## 2. Cross-fitted validation design

Apply the same fold-specific workflow to odd/even and chronological split-half analyses.

### Cross-fitted design

1. Split participants into 80% training and 20% held-out folds.
2. Fit scaling and MMER-XGBoost using only free-living meals from training participants.
3. Freeze scalers, the fixed-effect model, covariance components, and PC basis.
4. Divide held-out participants' meals into non-overlapping odd/even or chronological halves.
5. Transform and score both halves with the frozen training-fold model.
6. Compare held-out PC estimates and export fold-specific and pooled agreement statistics.

## 3. Held-out split-half analysis

Prepare the eligible analysis sample, define the fold-level fitting and scoring functions, and run odd/even and chronological splits. The displayed score tables, diagnostics, and ICC summaries are calculated from held-out participants only.

In [5]:
import os
import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from sklearn.model_selection import GroupKFold
from scipy.stats import pearsonr, spearmanr

import xgboost as xgb
from mmer import MixedEffectEstimator

In [6]:
N_FOLDS = 5

# Number of folds fitted simultaneously.
# With 4 threads inside each XGBoost/MMER model, start with 2 outer jobs.
N_JOBS_OUTER = 4
N_JOBS_MODEL = 8

RANDOM_SLOPES = [
    "carb_eaten",
    "fat_eaten",
    "protein_eaten",
    "fiber_eaten",
]

OUTCOMES = [
    "max_glucose",
    "peak_duration",
    "end_glucose",
    "positive_iAUC",
]

SUBJECT_COL = "subject_key"
DATE_COL = "eaten_date"

MIN_TOTAL_MEALS = 1
MIN_MEALS_PER_HALF = 1

In [7]:
def prepare_analysis_data(df_food):

    data[DATE_COL] = pd.to_datetime(data[DATE_COL])

    # Keep participants with enough total meals for reliable two-half inference.
    meal_counts = data.groupby(SUBJECT_COL).size()
    eligible_subjects = meal_counts[
        meal_counts >= MIN_TOTAL_MEALS
    ].index

    data = data[
        data[SUBJECT_COL].isin(eligible_subjects)
    ].copy()

    data = (
        data
        .sort_values([SUBJECT_COL, DATE_COL])
        .reset_index(drop=True)
    )

    return data

In [8]:

print("Meals:", len(data))
print("Participants:", data[SUBJECT_COL].nunique())

Meals: 54987
Participants: 992


In [9]:
def process_one_fold(
    fold_id,
    train_indices,
    test_indices,
    data,
    split_mode="even_odd",
    n_jobs_model=4,
    min_meals_per_half=8,
):
    """
    Fit one fold and produce two independently inferred PC scores
    for every eligible held-out participant.
    """
    train_subjects = data.iloc[train_indices][SUBJECT_COL].unique()
    test_subjects = data.iloc[test_indices][SUBJECT_COL].unique()

    # GroupKFold indexes participants; select meal rows by participant identifier.
    train_data = data[
        data[SUBJECT_COL].isin(train_subjects)
    ].copy()

    test_data = data[
        data[SUBJECT_COL].isin(test_subjects)
    ].copy()

    overlap = set(train_subjects).intersection(test_subjects)

    if overlap:
        raise RuntimeError(
            f"Participant leakage detected in fold {fold_id}."
        )

    print(
        f"Fold {fold_id}: "
        f"{len(train_subjects)} training participants, "
        f"{len(test_subjects)} held-out participants"
    )

    # 1. Fit using training participants only.
    fitted_result, fold_scalers, train_info = fit_training_fold(
        train_data=train_data,
        n_jobs_model=n_jobs_model,
    )

    # 2. Define PCs from training covariance only.
    eigenvalues, eigenvectors = derive_pc_basis(
        fitted_result
    )

    # 3. Split held-out participants.
    half_A, half_B, span = split_half_within_subject(
        test_data,
        split_mode=split_mode,
        subject_col=SUBJECT_COL,
        date_col=DATE_COL,
        min_meals_per_half=min_meals_per_half,
    )

    # 4. Infer two BLUP estimates using the same frozen model.
    blup_A = infer_blups_from_frozen_model(
        fitted_result=fitted_result,
        validation_data=half_A,
        fold_scalers=fold_scalers,
    )

    blup_B = infer_blups_from_frozen_model(
        fitted_result=fitted_result,
        validation_data=half_B,
        fold_scalers=fold_scalers,
    )

    common_participants = (
        blup_A.index
        .intersection(blup_B.index)
    )

    blup_A = blup_A.loc[common_participants]
    blup_B = blup_B.loc[common_participants]

    # 5. Project both BLUP estimates onto identical fold-specific PCs.
    scores_A = project_blups_to_pcs(
        blups=blup_A,
        eigenvalues=eigenvalues,
        eigenvectors=eigenvectors,
        n_components=2,
        standardize=True,
    )

    scores_B = project_blups_to_pcs(
        blups=blup_B,
        eigenvalues=eigenvalues,
        eigenvectors=eigenvectors,
        n_components=2,
        standardize=True,
    )

    fold_scores = (
        scores_A.add_suffix("_A")
        .join(scores_B.add_suffix("_B"))
        .reset_index()
    )

    participant_column = (
        blup_A.index.name
        if blup_A.index.name is not None
        else "index"
    )

    fold_scores = fold_scores.rename(
        columns={participant_column: SUBJECT_COL}
    )

    fold_scores["fold"] = fold_id
    fold_scores["split_mode"] = split_mode

    # Attach meal counts.
    count_table = (
        span.pivot(
            index=SUBJECT_COL,
            columns="half",
            values="n_meals",
        )
        .rename(
            columns={
                "A": "n_meals_A",
                "B": "n_meals_B",
            }
        )
        .reset_index()
    )

    fold_scores = fold_scores.merge(
        count_table,
        on=SUBJECT_COL,
        how="left",
    )

    fold_diagnostics = {
        "fold": fold_id,
        "split_mode": split_mode,
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_scored_subjects": len(fold_scores),
        "n_train_meals": len(train_data),
        "n_test_meals": len(test_data),
        "converged": fitted_result.is_converged,
        "log_likelihood": fitted_result.best_log_likelihood,
        "eigenvalue_1": eigenvalues[0],
        "eigenvalue_2": eigenvalues[1],
        "pc1_variance_fraction": (
            eigenvalues[0] / eigenvalues.sum()
        ),
        "pc2_variance_fraction": (
            eigenvalues[1] / eigenvalues.sum()
        ),
    }

    loading_table = pd.DataFrame({
        "coordinate": blup_A.columns,
        "PC1": eigenvectors[:, 0],
        "PC2": eigenvectors[:, 1],
        "fold": fold_id,
        "split_mode": split_mode,
    })

    return {
        "scores": fold_scores,
        "diagnostics": fold_diagnostics,
        "loadings": loading_table,
    }

In [10]:
def process_subject_fold(
    fold_id,
    train_subjects,
    test_subjects,
    data,
    split_mode="even_odd",
    n_jobs_model=4,
    min_meals_per_half=5,
):
    train_data = data[
        data[SUBJECT_COL].isin(train_subjects)
    ].copy()

    test_data = data[
        data[SUBJECT_COL].isin(test_subjects)
    ].copy()

    if set(train_subjects).intersection(test_subjects):
        raise RuntimeError("Training/test participant leakage.")

    fitted_result, fold_scalers, train_info = fit_training_fold(
        train_data,
        n_jobs_model=n_jobs_model,
    )

    eigenvalues, eigenvectors = derive_pc_basis(
        fitted_result
    )

    half_A, half_B, span = split_half_within_subject(
        test_data,
        split_mode=split_mode,
        min_meals_per_half=min_meals_per_half,
    )

    blup_A = infer_blups_from_frozen_model(
        fitted_result,
        half_A,
        fold_scalers,
    )

    blup_B = infer_blups_from_frozen_model(
        fitted_result,
        half_B,
        fold_scalers,
    )

    common = blup_A.index.intersection(blup_B.index)
    blup_A = blup_A.loc[common]
    blup_B = blup_B.loc[common]

    scores_A = project_blups_to_pcs(
        blup_A,
        eigenvalues,
        eigenvectors,
        n_components=2,
        standardize=True,
    )

    scores_B = project_blups_to_pcs(
        blup_B,
        eigenvalues,
        eigenvectors,
        n_components=2,
        standardize=True,
    )

    scores = (
        scores_A.add_suffix("_A")
        .join(scores_B.add_suffix("_B"))
        .reset_index()
    )

    index_name = blup_A.index.name or "index"

    scores = scores.rename(
        columns={index_name: SUBJECT_COL}
    )

    scores["fold"] = fold_id
    scores["split_mode"] = split_mode

    counts = (
        span.pivot(
            index=SUBJECT_COL,
            columns="half",
            values="n_meals",
        )
        .rename(
            columns={
                "A": "n_meals_A",
                "B": "n_meals_B",
            }
        )
        .reset_index()
    )

    scores = scores.merge(
        counts,
        on=SUBJECT_COL,
        how="left",
    )

    diagnostics = {
        "fold": fold_id,
        "split_mode": split_mode,
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_scored_subjects": len(scores),
        "converged": fitted_result.is_converged,
        "log_likelihood": fitted_result.best_log_likelihood,
        "pc1_variance_fraction": eigenvalues[0] / eigenvalues.sum(),
        "pc2_variance_fraction": eigenvalues[1] / eigenvalues.sum(),
    }

    loadings = pd.DataFrame({
        "coordinate": blup_A.columns,
        "PC1": eigenvectors[:, 0],
        "PC2": eigenvectors[:, 1],
        "fold": fold_id,
        "split_mode": split_mode,
    })

    return {
        "scores": scores,
        "diagnostics": diagnostics,
        "loadings": loadings,
    }

In [11]:
def run_cross_fitted_analysis(
    data,
    split_mode="even_odd",
    n_splits=5,
    n_jobs_outer=2,
    n_jobs_model=4,
    min_meals_per_half=5,
):
    folds = make_subject_folds(
        data,
        n_splits=n_splits,
    )

    results = Parallel(
        n_jobs=n_jobs_outer,
        backend="loky",
        verbose=10,
    )(
        delayed(process_subject_fold)(
            fold_id=fold["fold_id"],
            train_subjects=fold["train_subjects"],
            test_subjects=fold["test_subjects"],
            data=data,
            split_mode=split_mode,
            n_jobs_model=n_jobs_model,
            min_meals_per_half=min_meals_per_half,
        )
        for fold in folds
    )

    score_table = pd.concat(
        [result["scores"] for result in results],
        ignore_index=True,
    )

    diagnostic_table = pd.DataFrame(
        [result["diagnostics"] for result in results]
    )

    loading_table = pd.concat(
        [result["loadings"] for result in results],
        ignore_index=True,
    )

    return {
        "scores": score_table,
        "diagnostics": diagnostic_table,
        "loadings": loading_table,
    }

In [ ]:
odd_even_result = run_cross_fitted_analysis(
    data=data,
    split_mode="even_odd",
    n_splits=N_FOLDS,
    n_jobs_outer=4,
    n_jobs_model=8,
    min_meals_per_half=MIN_MEALS_PER_HALF, #MIN_MEALS_PER_HALF = 1
)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
Finished: no further improvement!:  40%|████      | 20/50 05:49         
Finished: no further improvement!:  42%|████▏     | 21/50 06:01         
[Parallel(n_jobs=4)]: Done   2 out of   5 | elapsed:  6.5min remaining:  9.7min
Finished: no further improvement!:  62%|██████▏   | 31/50 09:07         
[Parallel(n_jobs=4)]: Done   3 out of   5 | elapsed:  9.7min remaining:  6.4min
Finished: no further improvement!:  62%|██████▏   | 31/50 09:38         
Running MMER Framework | Fitting Model ...:  44%|████▍     | 22/50 05:18

In [ ]:
odd_even_result["scores"]

In [ ]:
odd_even_result["diagnostics"]

In [12]:
odd_even_result["diagnostics"]

,fold,split_mode,n_train_subjects,n_test_subjects,n_scored_subjects,converged,log_likelihood,pc1_variance_fraction,pc2_variance_fraction
0,1,even_odd,793,199,199,True,-88429.275972,0.665357,0.130563
1,2,even_odd,793,199,199,True,-89583.285474,0.682171,0.120676
2,3,even_odd,794,198,198,True,-89469.668162,0.668710,0.115054
3,4,even_odd,794,198,198,True,-88724.091488,0.700232,0.110309
4,5,even_odd,794,198,198,True,-90223.928526,0.672933,0.115917


In [13]:
print("odd/even split estimations :")
summary, full = icc_split_half(odd_even_result["scores"])
print(summary.to_string(index=False))

odd/even split estimations :
  fold  pc   n  ICC(A,1)  ICC(A,1) CI  ICC(C,1)  ICC(C,1) CI  pval    r         r CI  rho       rho CI
     1 PC1 199      0.84 [0.79, 0.87]      0.84 [0.79, 0.87]   0.0 0.84 [0.79, 0.87] 0.81 [0.75, 0.85]
     1 PC2 199      0.48 [0.36, 0.58]      0.48 [0.36, 0.58]   0.0 0.48 [0.36, 0.58] 0.45 [0.34, 0.56]
     2 PC1 199      0.81 [0.75, 0.85]      0.81 [0.75, 0.85]   0.0 0.81 [0.75, 0.85] 0.82 [0.77, 0.86]
     2 PC2 199      0.58 [0.48, 0.67]      0.58 [0.48, 0.67]   0.0 0.59 [0.49, 0.67] 0.48 [0.36, 0.58]
     3 PC1 198      0.83 [0.78, 0.87]      0.83 [0.78, 0.87]   0.0 0.83 [0.78, 0.87] 0.78 [0.72, 0.83]
     3 PC2 198      0.49 [0.37, 0.59]      0.49 [0.37, 0.58]   0.0 0.49 [0.37, 0.59] 0.44 [0.32, 0.54]
     4 PC1 198      0.80 [0.74, 0.85]      0.80 [0.74, 0.85]   0.0 0.80 [0.74, 0.85] 0.78 [0.72, 0.83]
     4 PC2 198      0.53 [0.42, 0.62]      0.53 [0.42, 0.62]   0.0 0.54 [0.43, 0.63] 0.49 [0.38, 0.59]
     5 PC1 198      0.80 [0.74, 0.85]      0

In [ ]:
summary.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_odd_even_split_half_summary.csv")

full.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_odd_even_split_half_full.csv")

In [ ]:
temporal_split_result = run_cross_fitted_analysis(
    data=data,
    split_mode="temporal",
    n_splits=N_FOLDS,
    n_jobs_outer=4,
    n_jobs_model=8,
    min_meals_per_half=MIN_MEALS_PER_HALF,
)

In [14]:
print("temporal split estimations :")
summary, full = icc_split_half(temporal_split_result["scores"])
print(summary.to_string(index=False))

temporal split estimations :
  fold  pc   n  ICC(A,1)  ICC(A,1) CI  ICC(C,1)  ICC(C,1) CI  pval    r         r CI  rho       rho CI
     1 PC1 199      0.73 [0.66, 0.79]      0.73 [0.66, 0.79]   0.0 0.73 [0.66, 0.79] 0.71 [0.63, 0.77]
     1 PC2 199      0.27 [0.14, 0.40]      0.28 [0.15, 0.40]   0.0 0.28 [0.15, 0.40] 0.31 [0.18, 0.43]
     2 PC1 199      0.72 [0.64, 0.78]      0.72 [0.64, 0.78]   0.0 0.72 [0.64, 0.78] 0.71 [0.64, 0.78]
     2 PC2 199      0.61 [0.51, 0.69]      0.61 [0.51, 0.69]   0.0 0.61 [0.51, 0.69] 0.48 [0.36, 0.58]
     3 PC1 198      0.78 [0.72, 0.83]      0.78 [0.72, 0.83]   0.0 0.79 [0.73, 0.84] 0.69 [0.61, 0.76]
     3 PC2 198      0.46 [0.34, 0.56]      0.46 [0.34, 0.56]   0.0 0.46 [0.35, 0.57] 0.41 [0.29, 0.52]
     4 PC1 198      0.73 [0.66, 0.79]      0.73 [0.65, 0.79]   0.0 0.73 [0.66, 0.79] 0.68 [0.60, 0.75]
     4 PC2 198      0.52 [0.41, 0.61]      0.52 [0.41, 0.61]   0.0 0.52 [0.41, 0.62] 0.50 [0.39, 0.60]
     5 PC1 198      0.73 [0.66, 0.79]      0

In [15]:
temporal_split_result["diagnostics"]

,fold,split_mode,n_train_subjects,n_test_subjects,n_scored_subjects,converged,log_likelihood,pc1_variance_fraction,pc2_variance_fraction
0,1,temporal,793,199,199,True,-88429.275972,0.665357,0.130563
1,2,temporal,793,199,199,True,-89583.285474,0.682171,0.120676
2,3,temporal,794,198,198,True,-89469.668162,0.668710,0.115054
3,4,temporal,794,198,198,True,-88724.091488,0.700232,0.110309
4,5,temporal,794,198,198,True,-90223.928526,0.672933,0.115917


In [16]:
summary.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_temporal_split_half_summary.csv")

full.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_temporal_split_half_full.csv")